# NSP10-NSP14 Interface Analysis
## Pan-Coronavirus RTC Inhibitor Discovery
**Candidate:** Olivier Nsekuye | University of Liège GIGA-VIN Lab

This notebook visualizes:
1. Interface contact map across 7DIY, 5C8T, AF3
2. Hotspot scores bar chart
3. Conservation heatmap (5 coronaviruses × interface residues)
4. 3D structure viewer — HIS80-ASP126 salt bridge highlighted
5. Pocket summary

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import py3Dmol
from pathlib import Path
from Bio import PDB

PROJECT = Path.home() / 'projects' / 'rtc-pan-coronavirus'
RES_DIR = PROJECT / '02-validation' / 'NSP10-NSP14'
PDB_DIR = PROJECT / '00-reference' / 'pdb_structures'
AF3_DIR = PROJECT / '01-alphafold3' / 'NSP10-NSP14'

print('Paths OK')

## 1. Load All Results

In [ ]:
# Load interface analysis
with open(RES_DIR / 'interface_analysis.json') as f:
    interface = json.load(f)

# Load conservation
cons_nsp10 = pd.read_csv(RES_DIR / 'conservation_NSP10.csv')
cons_nsp14 = pd.read_csv(RES_DIR / 'conservation_NSP14.csv')

# Load validation
with open(RES_DIR / 'validation_result.json') as f:
    validation = json.load(f)

# Load pocket analysis
with open(RES_DIR / 'pocket_analysis_2.json') as f:
    pockets = json.load(f)

print('Interface structures:', list(interface.keys()))
print('NSP10 conservation rows:', len(cons_nsp10))
print('NSP14 conservation rows:', len(cons_nsp14))
print('Validation F1:', validation.get('overall_f1', 'N/A'))

## 2. Hotspot Score Bar Charts

In [ ]:
# Conserved hotspot sets
CONS_NSP10 = {5, 19, 21, 40, 42, 44, 45, 80, 93}
CONS_NSP14 = {4, 7, 8, 9, 10, 20, 25, 27, 127, 201}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle('NSP10-NSP14 Interface Hotspot Conservation\nPan-Coronavirus Analysis (5 coronaviruses)',
             fontsize=14, fontweight='bold', y=1.02)

for ax, df, label, cons_set, color_main, color_high in [
    (axes[0], cons_nsp10, 'NSP10 (Chain A)', CONS_NSP10, '#4C72B0', '#C44E52'),
    (axes[1], cons_nsp14, 'NSP14 (Chain B)', CONS_NSP14, '#4C72B0', '#C44E52'),
]:
    hdf = df[df['is_hotspot']].sort_values('position')
    colors = ['#C44E52' if r in cons_set else '#4C72B0'
              for r in hdf['position']]
    bars = ax.barh(range(len(hdf)), hdf['conservation'],
                   color=colors, edgecolor='white', linewidth=0.5)
    ax.set_yticks(range(len(hdf)))
    ax.set_yticklabels(
        [f"{row['aa_SARS2']}{int(row['position'])}" for _, row in hdf.iterrows()],
        fontsize=10)
    ax.axvline(0.8, color='black', linestyle='--', linewidth=1.2,
               label='Conservation threshold (0.8)')
    ax.set_xlim(0, 1.05)
    ax.set_xlabel('Conservation Score', fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    # Add score labels
    for i, (_, row) in enumerate(hdf.iterrows()):
        ax.text(row['conservation'] + 0.01, i,
                f"{row['conservation']:.2f}",
                va='center', fontsize=8)

    conserved = mpatches.Patch(color='#C44E52', label='Conserved ≥0.8')
    variable  = mpatches.Patch(color='#4C72B0', label='Variable <0.8')
    ax.legend(handles=[conserved, variable], loc='lower right', fontsize=9)

plt.tight_layout()
out = PROJECT / 'results' / 'NSP10-NSP14_hotspot_conservation_2.png'
out.parent.mkdir(exist_ok=True)
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')

## 3. Conservation Heatmap — Hotspot Residues × Coronaviruses

In [ ]:
from Bio import AlignIO
from pathlib import Path

SEQ_DIR = PROJECT / '00-reference' / 'sequences' / 'conservation'
CORONAVIRUSES = ['SARS-CoV-2', 'SARS-CoV-1', 'MERS-CoV', 'HCoV-229E', 'HCoV-NL63']

CONS_HOTSPOTS = {
    'NSP10': [5, 19, 21, 40, 42, 44, 45, 80, 93],
    'NSP14': [4, 7, 8, 9, 10, 20, 25, 27, 127, 201]
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Amino Acid Conservation Across 5 Coronaviruses\nNSP10-NSP14 Conserved Hotspot Residues',
             fontsize=13, fontweight='bold')

for ax, nsp, df, hotspots in [
    (axes[0], 'NSP10', cons_nsp10, CONS_HOTSPOTS['NSP10']),
    (axes[1], 'NSP14', cons_nsp14, CONS_HOTSPOTS['NSP14']),
]:
    # Build AA matrix from alignment
    aln_file = SEQ_DIR / f'{nsp}_aligned.fasta'
    aln = AlignIO.read(aln_file, 'fasta')

    # Map alignment to reference (SARS-CoV-2) positions
    ref_seq = next(r for r in aln if 'SARS-CoV-2' in r.id)
    pos_map = {}
    ref_pos = 0
    for i, aa in enumerate(str(ref_seq.seq)):
        if aa != '-':
            ref_pos += 1
            pos_map[ref_pos] = i

    # Build matrix: rows=coronaviruses, cols=hotspot positions
    rows, row_labels = [], []
    for rec in aln:
        cov_name = rec.id.replace('_', '-')
        row = [str(rec.seq)[pos_map[p]] if p in pos_map else '-'
               for p in hotspots]
        rows.append(row)
        row_labels.append(cov_name)

    # Get SARS-CoV-2 AAs for column labels
    sars2_row = next(r for r in aln if 'SARS-CoV-2' in r.id)
    col_labels = [
        f"{str(sars2_row.seq)[pos_map[p]]}{p}" if p in pos_map else str(p)
        for p in hotspots
    ]

    # Conservation scores for color
    score_map = dict(zip(df['position'], df['conservation']))
    scores = np.array([[score_map.get(p, 0)] * len(rows)
                       for p in hotspots]).T

    im = ax.imshow(scores, cmap='RdYlGn', vmin=0, vmax=1,
                   aspect='auto')

    # Add AA text
    for i, row in enumerate(rows):
        for j, aa in enumerate(row):
            ax.text(j, i, aa, ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color='black')

    ax.set_xticks(range(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=10)
    ax.set_title(f'{nsp} Conserved Hotspots', fontsize=12, fontweight='bold')

    # Star the primary targets
    primary = {80, 93} if nsp == 'NSP10' else {126, 127}
    for j, p in enumerate(hotspots):
        if p in primary:
            ax.add_patch(plt.Rectangle((j-0.5, -0.5),
                         1, len(rows), fill=False,
                         edgecolor='blue', linewidth=2.5))

plt.colorbar(im, ax=axes, label='Conservation Score',
             shrink=0.6, pad=0.02)
plt.tight_layout()
out = PROJECT / 'results' / 'NSP10-NSP14_conservation_heatmap_2.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out.name}')
print('Blue boxes = primary drug target residues (HIS80, LYS93, ASP126, THR127)')

## 4. 3D Structure — HIS80-ASP126 Salt Bridge Highlighted

In [ ]:
pdb_path = PDB_DIR / '7DIY.pdb'
pdb_str  = pdb_path.read_text()

view = py3Dmol.view(width=900, height=600)
view.addModel(pdb_str, 'pdb')

# Cartoon base — NSP10 blue, NSP14 green
view.setStyle({'chain': 'A'}, {'cartoon': {'color': '#4C72B0', 'opacity': 0.85}})
view.setStyle({'chain': 'B'}, {'cartoon': {'color': '#55A868', 'opacity': 0.85}})

# Conserved hotspots NSP10 — orange sticks
cons_nsp10_res = [5,19,21,40,42,44,45,80,93]
for r in cons_nsp10_res:
    view.addStyle({'chain': 'A', 'resi': r},
                  {'stick': {'color': '#FF8C00', 'radius': 0.25}})

# Conserved hotspots NSP14 — yellow sticks
cons_nsp14_res = [4,7,8,9,10,20,25,27,127,201]
for r in cons_nsp14_res:
    view.addStyle({'chain': 'B', 'resi': r},
                  {'stick': {'color': '#FFD700', 'radius': 0.25}})

# PRIMARY: HIS80 — red sphere
view.addStyle({'chain': 'A', 'resi': 80},
              {'sphere': {'color': '#C44E52', 'radius': 1.2}})
view.addStyle({'chain': 'A', 'resi': 80},
              {'stick':  {'color': '#C44E52', 'radius': 0.4}})

# PRIMARY: ASP126 — red sphere
view.addStyle({'chain': 'B', 'resi': 126},
              {'sphere': {'color': '#C44E52', 'radius': 1.2}})
view.addStyle({'chain': 'B', 'resi': 126},
              {'stick':  {'color': '#C44E52', 'radius': 0.4}})

# LYS93 — magenta
view.addStyle({'chain': 'A', 'resi': 93},
              {'sphere': {'color': '#9B59B6', 'radius': 1.0}})
view.addStyle({'chain': 'A', 'resi': 93},
              {'stick':  {'color': '#9B59B6', 'radius': 0.35}})

# Labels
view.addLabel('HIS80 (NSP10)', {'position': {'chain': 'A', 'resi': 80},
    'backgroundColor': '#C44E52', 'fontColor': 'white',
    'fontSize': 12, 'backgroundOpacity': 0.85})
view.addLabel('ASP126 (NSP14)', {'position': {'chain': 'B', 'resi': 126},
    'backgroundColor': '#C44E52', 'fontColor': 'white',
    'fontSize': 12, 'backgroundOpacity': 0.85})
view.addLabel('LYS93 (NSP10)', {'position': {'chain': 'A', 'resi': 93},
    'backgroundColor': '#9B59B6', 'fontColor': 'white',
    'fontSize': 11, 'backgroundOpacity': 0.85})

# Zoom to interface
view.zoomTo({'chain': 'A', 'resi': [75, 85, 88, 98]})
view.show()

print('Legend:')
print('  Blue cartoon   = NSP10 (Chain A)')
print('  Green cartoon  = NSP14 (Chain B)')
print('  Red spheres    = HIS80 + ASP126 (primary salt bridge target)')
print('  Purple sphere  = LYS93 (secondary hotspot)')
print('  Orange sticks  = conserved NSP10 hotspots')
print('  Yellow sticks  = conserved NSP14 hotspots')

## 5. Docking Box Visualization

In [ ]:
# Show docking box on structure
box = {
    'center_x': -4.776, 'center_y': 7.298, 'center_z': -25.886,
    'size_x': 31.685,   'size_y': 34.288,  'size_z': 48.982
}

view2 = py3Dmol.view(width=900, height=600)
view2.addModel(pdb_str, 'pdb')
view2.setStyle({'chain': 'A'}, {'cartoon': {'color': '#4C72B0', 'opacity': 0.7}})
view2.setStyle({'chain': 'B'}, {'cartoon': {'color': '#55A868', 'opacity': 0.7}})

# Docking box
view2.addBox({
    'center': {'x': box['center_x'],
               'y': box['center_y'],
               'z': box['center_z']},
    'dimensions': {'w': box['size_x'],
                   'h': box['size_y'],
                   'd': box['size_z']},
    'color': 'orange',
    'opacity': 0.15,
    'wireframe': True
})

# HIS80 and ASP126 inside the box
view2.addStyle({'chain': 'A', 'resi': 80},
               {'sphere': {'color': '#C44E52', 'radius': 1.5}})
view2.addStyle({'chain': 'B', 'resi': 126},
               {'sphere': {'color': '#C44E52', 'radius': 1.5}})

view2.addLabel(f"Docking box\n{box['size_x']:.0f} x {box['size_y']:.0f} x {box['size_z']:.0f} Å",
    {'position': {'x': box['center_x'], 'y': box['center_y']+20,
                  'z': box['center_z']},
     'backgroundColor': 'orange', 'fontColor': 'black',
     'fontSize': 11, 'backgroundOpacity': 0.8})

view2.zoomTo()
view2.show()
print(f"Box center : ({box['center_x']}, {box['center_y']}, {box['center_z']})")
print(f"Box size   : {box['size_x']} x {box['size_y']} x {box['size_z']} Å")
print(f"Box volume : {box['size_x']*box['size_y']*box['size_z']:.0f} Å³")

## 6. Pipeline Summary

In [ ]:
print('=' * 55)
print('  NSP10-NSP14 Interface — Pipeline Summary')
print('=' * 55)
print()
print('AF3 Validation (Script 04):')
print(f'  Overall F1 = {validation.get("overall_f1", "N/A"):.3f}  ✅ PASS (gate: 0.70)')
print()
print('Interface Analysis (Script 05):')
print('  Primary target : HIS80(NSP10) — ASP126(NSP14) salt bridge')
print('  Distance (7DIY): 3.65 Å')
print('  Present in     : 7DIY ✅  5C8T ✅  AF3 ✅')
print()
print('Conservation (Script 06, 5 coronaviruses):')
nsp10_cons = cons_nsp10[cons_nsp10['is_hotspot'] & (cons_nsp10['conservation'] >= 0.8)]
nsp14_cons = cons_nsp14[cons_nsp14['is_hotspot'] & (cons_nsp14['conservation'] >= 0.8)]
print(f'  NSP10 conserved hotspots: {len(nsp10_cons)}')
print(f'  NSP14 conserved hotspots: {len(nsp14_cons)}')
his80 = cons_nsp10[cons_nsp10['position'] == 80]['conservation'].values
print(f'  HIS80 conservation : {his80[0]:.3f}  ✅ FULLY CONSERVED')
print()
print('Pocket Detection (Script 07_2):')
print('  Best pocket : 7DIY Pocket 3 (spans interface, contains HIS80)')
print('  Druggability: 0.000 (expected for PPI interface)')
print()
print('Docking Preparation (Script 08_2):')
print('  Receptor    : 7DIY chains A+B, 417 residues')
print('  Box center  : (-4.776, 7.298, -25.886) Å')
print('  Box size    : 31.7 x 34.3 x 49.0 Å')
print('  Box volume  : 53,215 Å³')
print('  Hotspots    : 9/9 NSP10 + 10/10 NSP14 verified ✅')
print()
print('STATUS: NSP10-NSP14 COMPLETE — Ready for virtual screening')
print('=' * 55)